# Phase 2 — Data Cleaning & Preprocessing

**Dataset:** Diabetes 130-US Hospitals (101,766 encounters)

Workflow: missing values → duplicates → quality assessment → outliers → feature engineering → encoding → save pipeline


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "backend"))

from app.ml.data_quality import generate_quality_report, missing_value_analysis, outlier_analysis
from app.ml.preprocessing import HealthcarePreprocessor, load_raw_data

sns.set_theme(style="whitegrid")
RAW_PATH = ROOT / "data" / "raw" / "diabetic_data.csv"
df = load_raw_data(RAW_PATH)
print(f"Shape: {df.shape}")
df.head()


## 1. Missing Value Analysis

The dataset uses `?` and `None` for missing values. Columns with **>40% missing** are dropped from modeling.


In [ ]:
missing = missing_value_analysis(df)
missing_df = pd.DataFrame(missing["by_column"]).T.sort_values("percent", ascending=False)
missing_df.head(10)


**Insight:** `weight` (97%), `max_glu_serum` (95%), and `A1Cresult` (83%) are too sparse for reliable modeling. `payer_code` and `medical_specialty` are also excluded.


## 2. Duplicates & Target Distribution


In [ ]:
report = generate_quality_report(df)
print("Duplicates:", report["duplicates"])
print("Target:", report["target_distribution"])

df["readmitted"].value_counts().plot(kind="bar", title="Readmission Outcomes", color="steelblue")
plt.ylabel("Count")
plt.tight_layout()
plt.show()


**Insight:** Only **11.16%** of patients are readmitted within 30 days — expect class imbalance (ratio ~4.8:1). No exact duplicate rows found; 30,248 repeat admissions are valid return visits.


## 3. Outlier Analysis (IQR)

Prior utilization columns show the most outliers — flagged but **not removed** (clinically valid high-utilization patients).


In [ ]:
outliers = outlier_analysis(df.replace("?", pd.NA))
outlier_df = pd.DataFrame(outliers).T[["max", "iqr_outlier_count", "iqr_outlier_percent"]]
outlier_df.sort_values("iqr_outlier_percent", ascending=False)


## 4. Run Preprocessing Pipeline


In [ ]:
preprocessor = HealthcarePreprocessor()
cleaned, feature_matrix = preprocessor.fit_transform(df)

print("Cleaned shape:", cleaned.shape)
print("ML matrix shape:", feature_matrix.shape)
print("Cleaning stats:", preprocessor.cleaning_stats)

cleaned.to_csv(ROOT / "data" / "processed" / "cleaned_data.csv", index=False)
preprocessor.save(ROOT / "models" / "preprocessor.joblib")
print("Saved cleaned_data.csv and preprocessor.joblib")


In [ ]:
cleaned[["age_midpoint", "time_in_hospital", "total_prior_visits", "readmitted_30_days"]].describe()
